In [ ]:
import shutil
import pandas as pd
import json

from pathlib import Path

In [ ]:
MILK10K_DOWNLOAD_PATH = Path("/home/sulcm/datasets/milk10k/milk10k_downloaded")
MILK10K_TRAIN = Path("/home/sulcm/datasets/milk10k/milk10k/train")

# Create dataset structure

In [ ]:
ds_gt = pd.read_csv(MILK10K_DOWNLOAD_PATH / "training_gt.csv").set_index("lesion_id", append=True)
labels = [l.lower() for l in ds_gt.columns.to_list()]
ds_gt_classes = ds_gt.dot(ds_gt.columns).apply(lambda x: x.lower())

In [ ]:
ds_training_input = pd.read_csv(MILK10K_DOWNLOAD_PATH / "milk10k" / "metadata.csv")
ds_training_input[["file_name", "label"]] = ds_training_input.apply(
    lambda row: [
        row["isic_id"] + ".jpg",
        ds_gt_classes.xs(row["lesion_id"], level=1).iloc[0]
    ],
    axis=1, result_type="expand"
)
ds_training_input.drop(columns=["attribution", "copyright_license"], inplace=True)

In [ ]:
ds_info = {
    "labels": labels
}

In [ ]:
# with open(MILK10K_TRAIN.parent / "dataset_info.json", "w") as f:
#     json.dump(ds_info, f, indent=2, ensure_ascii=False)

In [ ]:
if not MILK10K_TRAIN.exists():
    shutil.copytree(MILK10K_DOWNLOAD_PATH / "milk10k" / "images", MILK10K_TRAIN)
    ds_training_input.to_csv(MILK10K_TRAIN / "metadata.csv", index=False)
    with open(MILK10K_TRAIN.parent / "dataset_info.json", "w") as f:
        json.dump(ds_info, f, indent=2, ensure_ascii=False)

# Load/Build datatset

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("imagefolder", data_dir=MILK10K_TRAIN.parent)
dataset

In [ ]:
dataset["train"].info